# Сравнительный анализ алгоритмов ExtraTrees и RandomForest

## 1. Введение
Методы машинного обучения, основанные на решающих деревьях, широко применяются благодаря своей устойчивости к переобучению и способности улавливать нелинейные зависимости. Два наиболее популярных представителя - **Random Forest** (Случайный лес) и **Extra Trees** (Extremely Randomized Trees, Чрезвычайно случайные деревья). 

Несмотря на схожую архитектуру (оба строят множество деревьев), они используют разные подходы к формированию выборок и выбору порогов разбиения, что приводит к различиям в смещении, дисперсии и вычислительной сложности.

## 2. Теоретические различия
1. **Формирование обучающей выборки:**
   * **Random Forest:** Использует bootstrap - каждое дерево обучается на случайной подвыборке с возвращением. Это увеличивает разнообразие деревьев.
   * **Extra Trees:** Обучает каждое дерево на всей исходной выборке (без bootstrap).

2. **Выбор порога разбиения:**
   * **Random Forest:** Для каждого узла ищет **оптимальный** порог разбиения. Это требует перебора всех значений признаков, что делает алгоритм вычислительно затратным.
   * **Extra Trees:** Выбирает порог разбиения **случайным образом**. Для каждого признака берется случайное значение из его диапазона, и среди них выбирается лучшее. Это значительно ускоряет обучение.

## 3. Плюсы и минусы алгоритмов

### Random Forest
**Плюсы:**
* Высокая точность и отличная обобщающая способность.
* Хорошо работает с зашумленными данными и выбросами.
* Строит более гладкие и логичные границы решений (меньше склонность к переобучению на шуме).
* Предоставляет надежную оценку важности признаков.

**Минусы:**
* Относительно медленное обучение из-за поиска оптимальных порогов.
* Требует больше памяти.

### Extra Trees
**Плюсы:**
* **Высокая скорость обучения** (за счет отказа от поиска оптимальных порогов).
* Меньшая дисперсия (благодаря случайности порогов деревья получаются более независимыми).
* Отлично работает на очень больших датасетах.

**Минусы:**
* Более высокое смещение (bias). Может недообучаться на простых задачах.
* Может уступать Random Forest в точности на некоторых специфических данных.

## 3. Описание набора данных (Iris Dataset)

Для проведения сравнительного анализа используется классический набор данных **«Ирисы Фишера» (Iris Dataset)**.

**Характеристики датасета:**
* **Объем:** 150 образцов.
* **Классы:** 3 вида цветка ириса (по 50 образцов каждого). Классы сбалансированы.
  1. *Iris Setosa*
  2. *Iris Versicolor*
  3. *Iris Virginica*
* **Признаки (4 числовых непрерывных признака):**
  1. Длина чашелистика (Sepal Length) — в см.
  2. Ширина чашелистика (Sepal Width) — в см.
  3. Длина лепестка (Petal Length) — в см.
  4. Ширина лепестка (Petal Width) — в см.

**Почему выбран именно этот датасет?**
1. **Многоклассовость:** Позволяет оценить, как алгоритмы справляются с разделением более чем на 2 класса.
2. **Малая размерность:** Всего 4 признака, что позволяет легко визуализировать границы решений без применения методов понижения размерности.
3. **Наличие легкоразделимого класса:** *Iris Setosa* линейно отделим от двух других видов, в то время как *Versicolor* и *Virginica* пересекаются. Это отличный тест на способность алгоритма строить сложные нелинейные границы.

In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 5)

data = load_iris()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['target_name'] = df['target'].map({i: name for i, name in enumerate(data.target_names)})

display(Markdown("### Первые 5 строк датасета"))
display(df.head())

display(Markdown("### Общая информация о датасете"))
print(f"Размерность: {df.shape[0]} строк, {df.shape[1]} столбцов\n")
missing_data = df.isnull().sum().to_frame(name='Пропуски (NaN)')
display(missing_data)

display(Markdown("### Распределение классов"))
class_distribution = df['target_name'].value_counts().to_frame(name='Количество')
display(class_distribution)

display(Markdown("### Разделение выборки"))

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} объектов")
print(f"Тестовая выборка: {X_test.shape[0]} объектов")

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# n_estimators=100 означает, что лес будет состоять из 100 деревьев
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():    
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    cv_scores = cross_val_score(model, X, y, cv=5)
    
    results[name] = {
        "Accuracy (Test)": acc,
        "CV Mean Accuracy": cv_scores.mean(),
        "CV Std (Дисперсия)": cv_scores.std(),
        "Train Time (sec)": train_time
    }

display(Markdown("### Сравнительная таблица результатов"))

df_results = pd.DataFrame(results).T
df_results = df_results.round(4)
display(df_results)

### Анализ полученных метрик:
1. **Точность (Accuracy):** Оба алгоритма показывают высокий результат. На датасете Iris разница может быть минимальной, однако Random Forest часто дает чуть более высокую точность на тесте за счет поиска оптимальных разбиений.
2. **Время обучения (Train Time):** Здесь Extra Trees выигрывает. Отсутствие необходимости перебирать все возможные пороги делает его значительно быстрее. На больших данных эта разница становится критической.
3. **Кросс-валидация:** Средняя точность (CV Mean) подтверждает стабильность. Стандартное отклонение (CV Std) показывает разброс. У Extra Trees дисперсия обычно ниже, так как случайный выбор порогов делает деревья более независимыми друг от друга.

In [ ]:
for name, model in models.items():
    display(Markdown(f"### Отчет по классификации: {name}"))
    
    y_pred = model.predict(X_test)

    target_names = data.target_names
    report_dict = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
    df_report = pd.DataFrame(report_dict).transpose()

    display(df_report.round(2))

### Анализ отчетов по классификации

На основе полученных метрик (Precision, Recall, F1-score) можно сделать следующие выводы:

1. **Идеальная классификация (Iris Setosa):** Обе модели демонстрируют абсолютную точность (1.00) для класса *setosa*. Это объясняется тем, что данный вид цветка имеет ярко выраженные морфологические признаки (короткие и широкие лепестки) и линейно отделим от двух других классов.
2. **Сложность разделения (Versicolor и Virginica):** Основные ошибки обеих моделей сосредоточены в разделении *versicolor* и *virginica*. Признаки этих цветков частично пересекаются, что приводит к снижению метрик (до 0.78–0.93).
3. **Сравнение моделей:** 
   * У **Random Forest** наблюдается более высокий показатель *Recall* (полнота) для класса *virginica* (0.73), но при этом ниже *Precision* (точность) для *versicolor* (0.78). 
   * **Extra Trees** показал более сбалансированные результаты по классам, сгладив разницу между Precision и Recall (F1-score для virginica составил 0.86 против 0.81 у RF).

In [ ]:
# Визуализация границ решений на 2D плоскости
# Для наглядности возьмем только 2 признака: Длина лепестка (Petal Length) и Ширина лепестка (Petal Width)
feature_idx = [2, 3] 
X_2d = X[:, feature_idx]

def plot_decision_boundary(X, y, model, title, ax, is_trained=False):
    if not is_trained:
        model.fit(X, y)

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor='k', cmap=plt.cm.RdYlBu, s=50)
    feature_names = data.feature_names
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(feature_names[feature_idx[0]])
    ax.set_ylabel(feature_names[feature_idx[1]])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_2d = RandomForestClassifier(n_estimators=100, random_state=42)
et_2d = ExtraTreesClassifier(n_estimators=100, random_state=42)

plot_decision_boundary(X_2d, y, rf_2d, "Random Forest", axes[0])
plot_decision_boundary(X_2d, y, et_2d, "Extra Trees", axes[1])

plt.suptitle("Сравнение границ решений", fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

### Анализ границ решений

Визуализация на основе длины и ширины лепестка наглядно демонстрирует различия алгоритмов:

1. **Гладкость границ (Random Forest):** Границы решений у Random Forest выглядят более плавными и обобщенными. Алгоритм перебирает множество вариантов, чтобы найти оптимальный порог, который минимизирует ошибку. Это позволяет ему строить более точные границы в областях пересечения классов.
2. **Прямоугольность и хаотичность (Extra Trees):** Границы Extra Trees состоят из множества прямых углов и резких переходов. Это прямое следствие случайного выбора порогов разбиения. Деревья строятся быстрее, но не оптимизируют каждый шаг.
3. **Устойчивость к переобучению:** Несмотря на то, что границы Extra Trees выглядят «рваными» и менее интуитивными, в реальных задачах с сильным шумом такая рандомизация часто помогает модели меньше переобучаться на конкретные обучающие примеры, обеспечивая более низкую дисперсию.

## 4. Итоговые выводы

В ходе работы были обучены и протестированы две модели на основе деревьев решений. 

1. **Эффективность:** На данном датасете (Iris) обе модели показали себя отлично (точность > 0.93). Однако различия проявились в скорости и характере границ.
2. **Скорость против точности:** `Extra Trees` доказал свою вычислительную эффективность - время его обучения меньше. Это делает его предпочтительным выбором при работе с большими массивами данных, где ресурсы ограничены.
3. **Характер решений:** Визуализация подтвердила, что `Random Forest` ищет оптимальные пороги, создавая более гладкие границы. `Extra Trees` за счет рандомизации создает более "рваные" границы, что на сложных датасетах с шумом может служить защитой от переобучения (снижение дисперсии).
4. **Важность признаков:** Оба алгоритма сошлись во мнении, что ширина и длина лепестка (Petal Width/Length) являются наиболее информативными признаками для данной задачи.

**Итог:** Если задача требует максимальной точности и есть время на обучение - лучше выбрать *Random Forest*. Если важна скорость работы алгоритма и данные очень объемные - предпочтительнее *Extra Trees*.